# Anatomy-aware mRALE prediction with five MedGemma fold models

This notebook combines two completed training products:

1. The rank-32 anatomy/pathology localization LoRA identifies left- and right-lung regions.
2. The five rank-32 multitask fold adapters estimate mRALE from three evidence views: the whole CXR, a left-lung masked view, and a right-lung masked view.

The final method is constrained and auditable:

```text
CXR → anatomy LoRA → left/right lung boxes → whole/left/right evidence
    → fold mRALE adapter(s) → regional component consensus
    → right score + left score = final total mRALE
```

For each fold, regional views are preferred for their corresponding lung. If a regional output is invalid, the method falls back to the anatomy-conditioned whole-image output and then to the direct whole-image output. Across five models, integer extent and density components are combined by a robust median. The final total is always recomputed from the two lungs.

## Leakage-safe modes

- `out_of_fold` (default): evaluates the original five-fold dataset without leakage. Each image is predicted only by the adapter for the fold in which that image was held out. The five models are combined at the cross-validation result level.
- `external_ensemble`: runs all five fold adapters on every image and combines their predictions. Use this only for a dataset that was not used to train any fold adapter.

The notebook deliberately refuses an all-five ensemble on the internal fold records.


In [ ]:
# The successful training environment already contains Transformers and PEFT.
# Install evaluation dependencies if needed; restart the kernel afterward.
%pip install -U "pandas>=2.2" "scikit-learn>=1.5" "scipy>=1.12"


In [ ]:
import csv
import gc
import json
import math
import os
import random
import re
import statistics
from collections import Counter, defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageDraw
from peft import PeftModel
from scipy.stats import pearsonr, spearmanr, t
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from transformers import AutoModelForImageTextToText, AutoProcessor

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("BF16 supported:", torch.cuda.is_bf16_supported())


In [ ]:
# Configuration
BASE_MODEL_ID = "google/medgemma-1.5-4b-it"
ANATOMY_ADAPTER_DIR = Path(
    "/data/liangz2/openi/midrc/medgemma15_4b_cxr_bbox_lora_rank32/final_adapter"
)
FOLD_MODEL_ROOT = Path(
    "/data/liangz2/openi/midrc/medgemma15_4b_multitask_lora_rank32_cv"
)
FOLD_DATA_DIR = Path("/data/liangz2/openi/midrc/multi_task_CV")
OUTPUT_ROOT = Path(
    "/data/liangz2/openi/midrc/anatomy_aware_mrale_medgemma15_5fold"
)

FOLDS = [0, 1, 2, 3, 4]
FOLD_ADAPTER_PATTERN = "fold_{fold}/best_adapter"
FOLD_TRAIN_PATTERN = "multitask_train_fold_{fold}_harmony.jsonl"
FOLD_TEST_PATTERN = "multitask_test_fold_{fold}_harmony.jsonl"
IMAGE_PATH_REWRITES = {"/vf/users/liangz2/openi": "/data/liangz2/openi"}

EVALUATION_MODE = "out_of_fold"  # "out_of_fold" or "external_ensemble"
EXTERNAL_CSV = None               # Required for external_ensemble.
MAX_IMAGES = None                 # Use 8 for a smoke test, then None.
SEED = 42

COORDINATE_SCALE = 1000
LUNG_BOX_MARGIN_FRACTION = 0.04
ALLOW_HEURISTIC_BOX_FALLBACK = True
MAX_ANATOMY_NEW_TOKENS = 1000
MAX_MRALE_NEW_TOKENS = 320

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
assert EVALUATION_MODE in {"out_of_fold", "external_ensemble"}
assert ANATOMY_ADAPTER_DIR.joinpath("adapter_config.json").is_file(), ANATOMY_ADAPTER_DIR
for fold in FOLDS:
    adapter = FOLD_MODEL_ROOT / FOLD_ADAPTER_PATTERN.format(fold=fold)
    assert adapter.joinpath("adapter_config.json").is_file(), f"Missing fold adapter: {adapter}"
assert torch.cuda.is_available() and torch.cuda.is_bf16_supported(), "A BF16 CUDA GPU is required."

RUN_DIR = OUTPUT_ROOT / EVALUATION_MODE
RUN_DIR.mkdir(parents=True, exist_ok=True)
print("Mode:", EVALUATION_MODE)
print("Output:", RUN_DIR)


In [ ]:
# Load either leakage-safe out-of-fold records or a genuinely external CSV.
def rewrite_image_path(path):
    text = str(path)
    candidates = [Path(text)]
    for old_prefix, new_prefix in IMAGE_PATH_REWRITES.items():
        if text.startswith(old_prefix):
            candidates.append(Path(new_prefix + text[len(old_prefix):]))
    for candidate in candidates:
        if candidate.is_file():
            return str(candidate.resolve())
    raise FileNotFoundError(f"Image not found after path rewrite: {path}")


def read_jsonl(path):
    rows = []
    with Path(path).open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            if not line.strip():
                continue
            try:
                rows.append(json.loads(line))
            except Exception as error:
                raise ValueError(f"Invalid JSON at {path}:{line_number}") from error
    return rows


def exact_answer(record):
    ground_truth = record.get("ground_truth", {})
    answer = ground_truth.get("answer") if isinstance(ground_truth, dict) else None
    if not isinstance(answer, str):
        raise ValueError(f"Missing exact-string target for {record.get('id')}")
    return json.loads(answer)


def normalized_truth(target):
    er = int(target["extent_right_numerical"])
    dr = int(target["density_right_numerical"])
    el = int(target["extent_left_numerical"])
    dl = int(target["density_left_numerical"])
    total = float(target["mRALE Score"])
    right = er * dr
    left = el * dl
    if not math.isclose(total, right + left, abs_tol=1e-6):
        raise ValueError(f"Ground-truth components do not sum: {target}")
    return {
        "extent_right_numerical": er,
        "density_right_numerical": dr,
        "extent_left_numerical": el,
        "density_left_numerical": dl,
        "mrale_right": right,
        "mrale_left": left,
        "mrale_total": total,
    }


def load_out_of_fold_examples():
    examples = []
    for fold in FOLDS:
        path = FOLD_DATA_DIR / FOLD_TEST_PATTERN.format(fold=fold)
        if not path.is_file():
            raise FileNotFoundError(path)
        records = [record for record in read_jsonl(path) if record.get("task") == "mrale_prediction"]
        for record in records:
            examples.append({
                "id": str(record.get("file_name") or record["id"]),
                "record_id": record["id"],
                "image_path": rewrite_image_path(record["image_path"]),
                "held_out_fold": fold,
                "ground_truth": normalized_truth(exact_answer(record)),
            })
    ids = [example["id"] for example in examples]
    if len(ids) != len(set(ids)):
        duplicates = [item for item, count in Counter(ids).items() if count > 1]
        raise ValueError(f"OOF mRALE records are not unique by image: {duplicates[:10]}")
    return examples


def load_external_examples(path):
    if path is None:
        raise ValueError("Set EXTERNAL_CSV for external_ensemble mode.")
    path = Path(path)
    required = {
        "filename", "path", "extent_right_numerical", "density_right_numerical",
        "extent_left_numerical", "density_left_numerical", "mRALE Score",
    }
    with path.open("r", encoding="utf-8-sig", newline="") as handle:
        reader = csv.DictReader(handle)
        missing = required - set(reader.fieldnames or [])
        if missing:
            raise ValueError(f"External CSV is missing: {sorted(missing)}")
        rows = list(reader)
    examples = []
    for row in rows:
        target = {
            "extent_right_numerical": row["extent_right_numerical"],
            "density_right_numerical": row["density_right_numerical"],
            "extent_left_numerical": row["extent_left_numerical"],
            "density_left_numerical": row["density_left_numerical"],
            "mRALE Score": row["mRALE Score"],
        }
        examples.append({
            "id": row["filename"].strip(),
            "record_id": row["filename"].strip(),
            "image_path": rewrite_image_path(row["path"]),
            "held_out_fold": None,
            "ground_truth": normalized_truth(target),
        })
    if len({example["id"] for example in examples}) != len(examples):
        raise ValueError("External CSV filenames must be unique.")
    internal_image_ids = set()
    for fold in FOLDS:
        for pattern in [FOLD_TRAIN_PATTERN, FOLD_TEST_PATTERN]:
            fold_path = FOLD_DATA_DIR / pattern.format(fold=fold)
            if fold_path.is_file():
                internal_image_ids.update(
                    str(record.get("file_name") or Path(record.get("image_path", "")).name)
                    for record in read_jsonl(fold_path)
                )
    overlap = {example["id"] for example in examples} & internal_image_ids
    if overlap:
        raise ValueError(
            f"External ensemble leakage guard: {len(overlap)} external IDs occur in fold data; "
            f"examples={sorted(overlap)[:10]}. Use out_of_fold mode for internal MIDRC images."
        )
    return examples


examples = (
    load_out_of_fold_examples()
    if EVALUATION_MODE == "out_of_fold"
    else load_external_examples(EXTERNAL_CSV)
)
examples.sort(key=lambda item: item["id"])
if MAX_IMAGES is not None:
    examples = examples[:MAX_IMAGES]
if not examples:
    raise RuntimeError("No evaluation images were loaded.")

pd.DataFrame([
    {
        "id": example["id"], "image_path": example["image_path"],
        "held_out_fold": example["held_out_fold"], **example["ground_truth"],
    }
    for example in examples
]).to_csv(RUN_DIR / "evaluation_manifest.csv", index=False)
print("Images:", len(examples))
print("Per held-out fold:", dict(sorted(Counter(e["held_out_fold"] for e in examples).items(), key=str)))


In [ ]:
# Prompts retain the schemas used by the two successful training notebooks.
ANATOMY_SYSTEM_PROMPT = (
    "You are a chest radiograph localization assistant. Identify the annotated thoracic anatomy. "
    "Return only valid JSON with coordinate_system and anatomy. Each anatomy item must contain "
    "region, laterality, and bbox. bbox is [x1,y1,x2,y2] in a 0-1000 coordinate system relative "
    "to the full image, with the origin at the upper-left."
)
ANATOMY_USER_PROMPT = "Localize the annotated thoracic anatomy in this frontal chest X-ray."

MRALE_SYSTEM_PROMPT = (
    "You are a radiology assistant trained to evaluate frontal chest X-rays using the modified "
    "Radiographic Assessment of Lung Edema (mRALE) framework. Assess each lung independently and "
    "return only valid JSON with exactly these keys in this order: extent_right, density_right, "
    "extent_left, density_left, extent_right_numerical, density_right_numerical, "
    "extent_left_numerical, density_left_numerical, mRALE Score. Extent is an integer 0-4, density "
    "is an integer 0-3, each lung score is extent multiplied by density, and the total is the sum."
)
DIRECT_USER_PROMPT = (
    "Predict the qualitative and numerical involvement and density scores for both lungs and the "
    "total mRALE Score."
)


def anatomy_aware_user_prompt(left_box, right_box):
    return (
        "Use the image and the anatomy localizer's normalized 0-1000 lung regions to assess each lung "
        f"separately. Patient left lung bbox={left_box}; patient right lung bbox={right_box}. "
        "Reason internally about opacity extent and density within each lung, then return only the "
        "required JSON and ensure the total equals the two regional products."
    )


def regional_user_prompt(side, box):
    return (
        f"This image preserves the original PA CXR canvas but only the localized patient {side} lung "
        f"region is visible (normalized bbox={box}). Estimate the {side}-lung extent and density. "
        "Return the complete required mRALE JSON schema; only the numerical fields for the visible "
        f"{side} lung will be used by the constrained reasoning stage."
    )


def multimodal_messages(system_prompt, user_prompt):
    return [
        {"role": "system", "content": [{"type": "text", "text": system_prompt}]},
        {"role": "user", "content": [
            {"type": "image"},
            {"type": "text", "text": user_prompt},
        ]},
    ]


In [ ]:
# Model, generation, and checkpoint helpers.
processor = AutoProcessor.from_pretrained(BASE_MODEL_ID)
if processor.tokenizer.pad_token_id is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token
processor.tokenizer.padding_side = "left"


def load_adapter_model(adapter_dir):
    base = AutoModelForImageTextToText.from_pretrained(
        BASE_MODEL_ID,
        torch_dtype=torch.bfloat16,
        device_map="auto",
        low_cpu_mem_usage=True,
    )
    base.config.use_cache = True
    model = PeftModel.from_pretrained(base, str(adapter_dir), is_trainable=False)
    model.eval()
    return model


def model_input_device(model):
    for parameter in model.parameters():
        if parameter.device.type not in {"meta", "cpu"}:
            return parameter.device
    return torch.device("cuda:0")


def extract_json_object(text):
    cleaned = re.sub(r"^```(?:json)?\s*", "", text.strip(), flags=re.IGNORECASE)
    cleaned = re.sub(r"\s*```$", "", cleaned)
    start = cleaned.find("{")
    if start < 0:
        raise ValueError("No JSON object found")
    obj, _ = json.JSONDecoder().raw_decode(cleaned[start:])
    if not isinstance(obj, dict):
        raise ValueError("Prediction is not a JSON object")
    return obj


@torch.inference_mode()
def generate(model, image_path, system_prompt, user_prompt, max_new_tokens):
    prompt = processor.apply_chat_template(
        multimodal_messages(system_prompt, user_prompt),
        add_generation_prompt=True,
        tokenize=False,
    )
    with Image.open(image_path) as image_file:
        image = image_file.convert("RGB")
        inputs = processor(text=prompt, images=image, return_tensors="pt")
    device = model_input_device(model)
    moved = {
        key: value.to(device=device, dtype=torch.bfloat16) if value.is_floating_point() else value.to(device)
        for key, value in inputs.items()
    }
    prompt_length = moved["input_ids"].shape[-1]
    output_ids = model.generate(
        **moved,
        do_sample=False,
        max_new_tokens=max_new_tokens,
        pad_token_id=processor.tokenizer.pad_token_id,
        eos_token_id=processor.tokenizer.eos_token_id,
        use_cache=True,
    )
    return processor.decode(output_ids[0, prompt_length:], skip_special_tokens=True).strip()


def safe_parse(text):
    try:
        return extract_json_object(text), None
    except Exception as error:
        return None, str(error)


def load_jsonl_by_key(path, key_fields):
    rows = {}
    if not path.is_file():
        return rows
    with path.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            if not line.strip():
                continue
            try:
                row = json.loads(line)
                key = tuple(str(row[field]) for field in key_fields)
                rows[key] = row
            except Exception as error:
                print(f"Ignoring malformed {path.name} line {line_number}: {error}")
    return rows


def append_jsonl(path, row):
    with path.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(row, ensure_ascii=False) + "\n")
        handle.flush()
        os.fsync(handle.fileno())


def release_model(model):
    del model
    gc.collect()
    torch.cuda.empty_cache()


In [ ]:
# Stage 1: localize left/right lungs and create orientation-preserving masked views.
def valid_box(box):
    try:
        x1, y1, x2, y2 = [float(value) for value in box]
        return 0 <= x1 < x2 <= COORDINATE_SCALE and 0 <= y1 < y2 <= COORDINATE_SCALE
    except (TypeError, ValueError):
        return False


def normalized_laterality(item):
    laterality = str(item.get("laterality", "")).strip().lower()
    region = str(item.get("region", "")).strip().lower()
    if laterality in {"left", "right"}:
        return laterality
    if "left" in region:
        return "left"
    if "right" in region:
        return "right"
    return None


def union_boxes(boxes):
    return [
        min(box[0] for box in boxes), min(box[1] for box in boxes),
        max(box[2] for box in boxes), max(box[3] for box in boxes),
    ]


def extract_lung_boxes(parsed):
    candidates = {"left": [], "right": []}
    anatomy = parsed.get("anatomy", []) if isinstance(parsed, dict) else []
    for item in anatomy:
        if not isinstance(item, dict) or not valid_box(item.get("bbox")):
            continue
        region = str(item.get("region", "")).lower()
        if "lung" not in region or "hil" in region:
            continue
        laterality = normalized_laterality(item)
        if laterality:
            candidates[laterality].append([float(value) for value in item["bbox"]])
    boxes = {
        side: union_boxes(side_boxes) if side_boxes else None
        for side, side_boxes in candidates.items()
    }
    sources = {side: "anatomy_lora" if boxes[side] is not None else None for side in boxes}
    heuristic = {
        # On a standard PA display, patient left is on the image's right.
        "left": [450, 80, 1000, 970],
        "right": [0, 80, 550, 970],
    }
    for side in ["left", "right"]:
        if boxes[side] is None:
            if not ALLOW_HEURISTIC_BOX_FALLBACK:
                raise ValueError(f"No {side} lung box returned by anatomy model")
            boxes[side] = heuristic[side]
            sources[side] = "heuristic_fallback"
    return boxes, sources


def expand_box(box, margin_fraction=LUNG_BOX_MARGIN_FRACTION):
    x1, y1, x2, y2 = box
    width, height = x2 - x1, y2 - y1
    return [
        max(0, x1 - margin_fraction * width),
        max(0, y1 - margin_fraction * height),
        min(COORDINATE_SCALE, x2 + margin_fraction * width),
        min(COORDINATE_SCALE, y2 + margin_fraction * height),
    ]


def masked_view(image_path, normalized_box, output_path):
    with Image.open(image_path) as source:
        image = source.convert("RGB")
    width, height = image.size
    x1, y1, x2, y2 = expand_box(normalized_box)
    pixel_box = (
        max(0, round(x1 * width / COORDINATE_SCALE)),
        max(0, round(y1 * height / COORDINATE_SCALE)),
        min(width, round(x2 * width / COORDINATE_SCALE)),
        min(height, round(y2 * height / COORDINATE_SCALE)),
    )
    canvas = Image.new("RGB", image.size, (0, 0, 0))
    canvas.paste(image.crop(pixel_box), pixel_box[:2])
    output_path.parent.mkdir(parents=True, exist_ok=True)
    canvas.save(output_path)
    return str(output_path)


localization_path = RUN_DIR / "anatomy_localizations.jsonl"
localizations = load_jsonl_by_key(localization_path, ["id"])
anatomy_model = load_adapter_model(ANATOMY_ADAPTER_DIR)

for index, example in enumerate(examples, start=1):
    key = (example["id"],)
    if key in localizations:
        continue
    raw = generate(
        anatomy_model, example["image_path"], ANATOMY_SYSTEM_PROMPT,
        ANATOMY_USER_PROMPT, MAX_ANATOMY_NEW_TOKENS,
    )
    parsed, parse_error = safe_parse(raw)
    boxes, sources = extract_lung_boxes(parsed or {})
    view_dir = RUN_DIR / "masked_lung_views"
    left_path = masked_view(example["image_path"], boxes["left"], view_dir / f"{example['id']}__left.png")
    right_path = masked_view(example["image_path"], boxes["right"], view_dir / f"{example['id']}__right.png")
    row = {
        "id": example["id"], "image_path": example["image_path"],
        "raw_anatomy": raw, "parsed_anatomy": parsed, "parse_error": parse_error,
        "left_box": boxes["left"], "right_box": boxes["right"],
        "left_box_source": sources["left"], "right_box_source": sources["right"],
        "left_masked_image": left_path, "right_masked_image": right_path,
    }
    append_jsonl(localization_path, row)
    localizations[key] = row
    print(f"[anatomy] {index}/{len(examples)} {example['id']}")

del anatomy_model
gc.collect()
torch.cuda.empty_cache()
localization_fallback_rate = np.mean([
    any(localizations[(example["id"],)][f"{side}_box_source"] != "anatomy_lora" for side in ["left", "right"])
    for example in examples
])
print("Localization fallback rate:", localization_fallback_rate)


In [ ]:
# Stage 2: each eligible fold model predicts direct, anatomy-whole, left, and right views.
def eligible_examples_for_fold(fold):
    if EVALUATION_MODE == "external_ensemble":
        return examples
    return [example for example in examples if example["held_out_fold"] == fold]


def run_fold(fold):
    adapter_dir = FOLD_MODEL_ROOT / FOLD_ADAPTER_PATTERN.format(fold=fold)
    output_path = RUN_DIR / f"fold_{fold}_view_predictions.jsonl"
    completed = load_jsonl_by_key(output_path, ["id", "fold"])
    selected = eligible_examples_for_fold(fold)
    if not selected:
        print(f"[fold {fold}] no eligible images; skipping model load")
        return []
    model = load_adapter_model(adapter_dir)
    for index, example in enumerate(selected, start=1):
        key = (example["id"], str(fold))
        if key in completed:
            continue
        anatomy = localizations[(example["id"],)]
        requests = {
            "direct_whole": (
                example["image_path"], DIRECT_USER_PROMPT,
            ),
            "anatomy_whole": (
                example["image_path"], anatomy_aware_user_prompt(anatomy["left_box"], anatomy["right_box"]),
            ),
            "left_region": (
                anatomy["left_masked_image"], regional_user_prompt("left", anatomy["left_box"]),
            ),
            "right_region": (
                anatomy["right_masked_image"], regional_user_prompt("right", anatomy["right_box"]),
            ),
        }
        outputs = {}
        for view_name, (image_path, user_prompt) in requests.items():
            raw = generate(model, image_path, MRALE_SYSTEM_PROMPT, user_prompt, MAX_MRALE_NEW_TOKENS)
            parsed, parse_error = safe_parse(raw)
            outputs[view_name] = {"raw": raw, "parsed": parsed, "parse_error": parse_error}
        row = {
            "id": example["id"], "record_id": example["record_id"], "fold": fold,
            "held_out_fold": example["held_out_fold"], "image_path": example["image_path"],
            "ground_truth": example["ground_truth"], "views": outputs,
        }
        append_jsonl(output_path, row)
        completed[key] = row
        print(f"[fold {fold}] {index}/{len(selected)} {example['id']}")
    result_rows = [completed[(example["id"], str(fold))] for example in selected]
    del model
    gc.collect()
    torch.cuda.empty_cache()
    return result_rows


FOLD_ROWS = {}
for fold in FOLDS:
    FOLD_ROWS[fold] = run_fold(fold)


In [ ]:
# Normalize component evidence and apply the within-fold fallback hierarchy.
def bounded_integer(value, lower, upper):
    try:
        number = float(value)
    except (TypeError, ValueError):
        return None
    if not math.isfinite(number) or not lower <= number <= upper:
        return None
    return int(math.floor(number + 0.5))


def normalized_mrale(parsed):
    parsed = parsed if isinstance(parsed, dict) else {}
    result = {
        "er": bounded_integer(parsed.get("extent_right_numerical"), 0, 4),
        "dr": bounded_integer(parsed.get("density_right_numerical"), 0, 3),
        "el": bounded_integer(parsed.get("extent_left_numerical"), 0, 4),
        "dl": bounded_integer(parsed.get("density_left_numerical"), 0, 3),
    }
    try:
        total = float(parsed.get("mRALE Score"))
        result["reported_total"] = total if math.isfinite(total) and 0 <= total <= 24 else None
    except (TypeError, ValueError):
        result["reported_total"] = None
    result["right_score"] = result["er"] * result["dr"] if result["er"] is not None and result["dr"] is not None else None
    result["left_score"] = result["el"] * result["dl"] if result["el"] is not None and result["dl"] is not None else None
    result["component_total"] = (
        result["right_score"] + result["left_score"]
        if result["right_score"] is not None and result["left_score"] is not None else None
    )
    return result


def choose_side(primary, anatomy_whole, direct_whole, side):
    fields = ("el", "dl") if side == "left" else ("er", "dr")
    for source_name, candidate in [
        (f"{side}_region", primary),
        ("anatomy_whole", anatomy_whole),
        ("direct_whole", direct_whole),
    ]:
        if all(candidate[field] is not None for field in fields):
            return {"extent": candidate[fields[0]], "density": candidate[fields[1]], "source": source_name}
    return {"extent": None, "density": None, "source": "invalid"}


def fold_evidence(row):
    views = {
        name: normalized_mrale(payload.get("parsed"))
        for name, payload in row["views"].items()
    }
    left = choose_side(views["left_region"], views["anatomy_whole"], views["direct_whole"], "left")
    right = choose_side(views["right_region"], views["anatomy_whole"], views["direct_whole"], "right")
    left_score = left["extent"] * left["density"] if left["extent"] is not None else None
    right_score = right["extent"] * right["density"] if right["extent"] is not None else None
    regional_total = left_score + right_score if left_score is not None and right_score is not None else None
    whole_total = views["anatomy_whole"]["reported_total"]
    if whole_total is None:
        whole_total = views["direct_whole"]["reported_total"]
    return {
        "fold": row["fold"], "left": left, "right": right,
        "left_score": left_score, "right_score": right_score,
        "regional_total": regional_total, "whole_total": whole_total,
        "direct": views["direct_whole"], "anatomy_whole": views["anatomy_whole"],
        "regional_vs_whole_gap": abs(regional_total - whole_total) if regional_total is not None and whole_total is not None else None,
    }


FOLD_EVIDENCE = defaultdict(list)
for fold, rows in FOLD_ROWS.items():
    for row in rows:
        FOLD_EVIDENCE[row["id"]].append(fold_evidence(row))

expected_models_per_image = 1 if EVALUATION_MODE == "out_of_fold" else len(FOLDS)
for example in examples:
    count = len(FOLD_EVIDENCE[example["id"]])
    if count != expected_models_per_image:
        raise RuntimeError(f"{example['id']}: expected {expected_models_per_image} fold outputs, found {count}")


In [ ]:
# Stage 3: robust cross-fold consensus and constrained reasoning.
def median_integer(values):
    valid = [value for value in values if value is not None]
    return int(math.floor(statistics.median(valid) + 0.5)) if valid else None


def median_float(values):
    valid = [float(value) for value in values if value is not None]
    return float(statistics.median(valid)) if valid else None


def iqr(values):
    valid = np.asarray([float(value) for value in values if value is not None])
    return float(np.percentile(valid, 75) - np.percentile(valid, 25)) if len(valid) else None


def consensus_from_view(evidence_list, view_name):
    views = [evidence[view_name] for evidence in evidence_list]
    er = median_integer([view["er"] for view in views])
    dr = median_integer([view["dr"] for view in views])
    el = median_integer([view["el"] for view in views])
    dl = median_integer([view["dl"] for view in views])
    right = er * dr if er is not None and dr is not None else None
    left = el * dl if el is not None and dl is not None else None
    total = right + left if right is not None and left is not None else median_float([view["reported_total"] for view in views])
    return {"er": er, "dr": dr, "el": el, "dl": dl, "mrale_right": right, "mrale_left": left, "mrale_total": total}


final_rows = []
for example in examples:
    evidence = sorted(FOLD_EVIDENCE[example["id"]], key=lambda item: item["fold"])
    er = median_integer([item["right"]["extent"] for item in evidence])
    dr = median_integer([item["right"]["density"] for item in evidence])
    el = median_integer([item["left"]["extent"] for item in evidence])
    dl = median_integer([item["left"]["density"] for item in evidence])
    right_score = er * dr if er is not None and dr is not None else None
    left_score = el * dl if el is not None and dl is not None else None
    total = right_score + left_score if right_score is not None and left_score is not None else None
    whole_consensus = median_float([item["whole_total"] for item in evidence])
    regional_totals = [item["regional_total"] for item in evidence]
    reasoning = (
        f"Anatomy localized patient left and right lungs. Across {len(evidence)} eligible fold model(s), "
        f"right extent={er}, right density={dr}, right score={right_score}; "
        f"left extent={el}, left density={dl}, left score={left_score}. "
        f"Regional sum={total}; whole-image consensus={whole_consensus}; "
        f"absolute consistency gap={abs(total - whole_consensus) if total is not None and whole_consensus is not None else None}. "
        "The final total is constrained to right score plus left score."
    )
    final_rows.append({
        "id": example["id"], "record_id": example["record_id"],
        "image_path": example["image_path"], "held_out_fold": example["held_out_fold"],
        "mode": EVALUATION_MODE, "ground_truth": example["ground_truth"],
        "anatomy": localizations[(example["id"],)],
        "eligible_fold_count": len(evidence), "fold_evidence": evidence,
        "direct_baseline": consensus_from_view(evidence, "direct"),
        "anatomy_whole": consensus_from_view(evidence, "anatomy_whole"),
        "anatomy_aware": {
            "extent_right_numerical": er, "density_right_numerical": dr,
            "extent_left_numerical": el, "density_left_numerical": dl,
            "mrale_right": right_score, "mrale_left": left_score, "mrale_total": total,
            "whole_image_total_consensus": whole_consensus,
            "regional_vs_whole_gap": abs(total - whole_consensus) if total is not None and whole_consensus is not None else None,
            "fold_regional_total_iqr": iqr(regional_totals),
            "reasoning": reasoning,
        },
    })

final_path = RUN_DIR / "anatomy_aware_predictions.jsonl"
with final_path.open("w", encoding="utf-8") as handle:
    for row in final_rows:
        handle.write(json.dumps(row, ensure_ascii=False) + "\n")
print("Saved:", final_path)


In [ ]:
# Evaluate direct, anatomy-conditioned whole-image, and anatomy-aware regional methods.
METHODS = ["direct_baseline", "anatomy_whole", "anatomy_aware"]
FIELD_MAP = {
    "mrale_total": ("mrale_total", 24.0),
    "mrale_right": ("mrale_right", 12.0),
    "mrale_left": ("mrale_left", 12.0),
}


def safe_corr(function, truth, prediction):
    if len(truth) < 2 or np.std(truth) == 0 or np.std(prediction) == 0:
        return float("nan")
    return float(function(truth, prediction).statistic)


def method_metrics(rows, method):
    result = {"method": method, "n": len(rows)}
    for output_field, (truth_field, invalid_penalty) in FIELD_MAP.items():
        truth = np.asarray([row["ground_truth"][truth_field] for row in rows], dtype=float)
        predictions = [row[method].get(output_field) for row in rows]
        valid = np.asarray([value is not None and math.isfinite(float(value)) for value in predictions])
        valid_prediction = np.asarray([float(value) for value in predictions if value is not None and math.isfinite(float(value))])
        valid_truth = truth[valid]
        prefix = output_field
        result[f"{prefix}_coverage"] = float(valid.mean())
        penalized = np.asarray([
            abs(float(prediction) - target) if is_valid else invalid_penalty
            for prediction, target, is_valid in zip(predictions, truth, valid)
        ])
        result[f"{prefix}_mae_penalized"] = float(penalized.mean())
        if len(valid_truth):
            result[f"{prefix}_mae_valid"] = float(mean_absolute_error(valid_truth, valid_prediction))
            result[f"{prefix}_rmse_valid"] = float(math.sqrt(mean_squared_error(valid_truth, valid_prediction)))
            result[f"{prefix}_within_1_valid"] = float(np.mean(np.abs(valid_prediction - valid_truth) <= 1))
            result[f"{prefix}_within_2_valid"] = float(np.mean(np.abs(valid_prediction - valid_truth) <= 2))
            result[f"{prefix}_pearson_valid"] = safe_corr(pearsonr, valid_truth, valid_prediction)
            result[f"{prefix}_spearman_valid"] = safe_corr(spearmanr, valid_truth, valid_prediction)
            result[f"{prefix}_r2_valid"] = float(r2_score(valid_truth, valid_prediction)) if len(valid_truth) >= 2 else float("nan")
    return result


overall_metrics = pd.DataFrame([method_metrics(final_rows, method) for method in METHODS])
overall_metrics.to_csv(RUN_DIR / "overall_metrics.csv", index=False)

per_fold_metrics = []
if EVALUATION_MODE == "out_of_fold":
    for fold in FOLDS:
        fold_rows = [row for row in final_rows if row["held_out_fold"] == fold]
        if not fold_rows:
            continue
        for method in METHODS:
            metrics = method_metrics(fold_rows, method)
            metrics["fold"] = fold
            per_fold_metrics.append(metrics)
    per_fold_frame = pd.DataFrame(per_fold_metrics)
    per_fold_frame.to_csv(RUN_DIR / "per_fold_metrics.csv", index=False)

    aggregate_rows = []
    numeric_columns = [column for column in per_fold_frame.columns if column not in {"fold", "method", "n"}]
    for method in METHODS:
        method_frame = per_fold_frame[per_fold_frame["method"] == method]
        for metric in numeric_columns:
            values = method_frame[metric].dropna().astype(float).to_numpy()
            if not len(values):
                continue
            mean = float(values.mean())
            sd = float(values.std(ddof=1)) if len(values) >= 2 else 0.0
            margin = float(t.ppf(0.975, len(values) - 1) * sd / math.sqrt(len(values))) if len(values) >= 2 else 0.0
            aggregate_rows.append({
                "method": method, "metric": metric, "n_folds": len(values),
                "mean": mean, "sample_std": sd,
                "ci95_lower": mean - margin, "ci95_upper": mean + margin,
            })
    pd.DataFrame(aggregate_rows).to_csv(RUN_DIR / "cross_fold_aggregate_95ci.csv", index=False)

overall_metrics


In [ ]:
# Save configuration and a small localization QA panel.
config = {
    "base_model_id": BASE_MODEL_ID,
    "anatomy_adapter_dir": str(ANATOMY_ADAPTER_DIR),
    "fold_model_root": str(FOLD_MODEL_ROOT),
    "folds": FOLDS,
    "mode": EVALUATION_MODE,
    "external_csv": str(EXTERNAL_CSV) if EXTERNAL_CSV is not None else None,
    "seed": SEED,
    "n_images": len(examples),
    "max_images": MAX_IMAGES,
    "localization_fallback_rate": float(localization_fallback_rate),
    "aggregation": "median integer extent/density; total constrained to right+left",
    "within_fold_fallback": ["regional view", "anatomy whole", "direct whole"],
    "leakage_policy": (
        "one held-out adapter per internal image"
        if EVALUATION_MODE == "out_of_fold" else "all five adapters; external data only"
    ),
}
with (RUN_DIR / "run_config.json").open("w", encoding="utf-8") as handle:
    json.dump(config, handle, indent=2)


def draw_lung_boxes(example, localization, max_side=1000):
    with Image.open(example["image_path"]) as source:
        image = source.convert("RGB")
    width, height = image.size
    draw = ImageDraw.Draw(image)
    for side, color in [("left", "#00D4FF"), ("right", "#FF3B30")]:
        box = localization[f"{side}_box"]
        pixel_box = [
            round(box[0] * width / COORDINATE_SCALE), round(box[1] * height / COORDINATE_SCALE),
            round(box[2] * width / COORDINATE_SCALE), round(box[3] * height / COORDINATE_SCALE),
        ]
        draw.rectangle(pixel_box, outline=color, width=max(3, width // 600))
        draw.text((pixel_box[0] + 5, pixel_box[1] + 5), side, fill=color)
    image.thumbnail((max_side, max_side))
    return image


print("Artifacts saved to:", RUN_DIR)
display(draw_lung_boxes(examples[0], localizations[(examples[0]["id"],)]))


## Interpretation

- The main comparison is `anatomy_aware` versus `direct_baseline`.
- `anatomy_whole` tests whether coordinates alone help; `anatomy_aware` additionally uses separate regional evidence views.
- Report the anatomy-localization fallback rate. If it is high, inspect localization outputs before interpreting mRALE changes.
- In out-of-fold mode, each internal image has exactly one eligible fold model. This is the unbiased estimate for the existing five-fold study.
- In external-ensemble mode, each image has five eligible models and the IQR of fold regional totals provides a simple uncertainty indicator.
- The textual reasoning is generated deterministically from the evidence and cannot alter the constrained score formula.
